# Capstone: Cryptocurrency Market Prediction
**Research question:** Can 15-minute price sequences combined with engineered momentum indicators and market metrics predict 24-hour directional price movement and returns for major cryptocurrencies?

Pipeline: load data → engineer features → build 24h target → chronological split → baselines → XGBoost → GRU → ensemble → permutation importance & SHAP.

> If no Kaggle file is found at `DATA_PATH`, the notebook generates **synthetic** prices so it runs end-to-end. Synthetic results are only a pipeline check, not evidence about real markets.

In [ ]:
import os, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score, mean_absolute_error
from sklearn.inspection import permutation_importance
from xgboost import XGBClassifier, XGBRegressor
warnings.filterwarnings("ignore")
RNG = 42
np.random.seed(RNG)

## 1. Configuration & data loading
Expected columns (case-insensitive, rename via `COLUMN_MAP`): timestamp, open, high, low, close, volume. For multi-asset files, set `ASSET_COL`/`ASSET` to filter one coin.

In [ ]:
DATA_PATH = "data/coin_price_15min.csv"   # e.g. Kaggle 'Coin Price Forecasting Challenge' file
COLUMN_MAP = {}                           # e.g. {"Open time": "timestamp"}
ASSET_COL, ASSET = None, None             # e.g. "Asset_ID", 1
BAR_MINUTES = 15
HORIZON_BARS = 24 * 60 // BAR_MINUTES     # 96 bars = 24h
SEQ_LEN = 32                              # bars per GRU window

def synthetic(n=40000):
    idx = pd.date_range("2021-01-01", periods=n, freq=f"{BAR_MINUTES}min")
    r = np.random.normal(0, 0.002, n) + 0.0004*np.sin(np.arange(n)/400)   # weak slow cycle
    close = 30000*np.exp(np.cumsum(r))
    high = close*(1+np.abs(np.random.normal(0, .001, n))); low = close*(1-np.abs(np.random.normal(0, .001, n)))
    open_ = np.r_[close[0], close[:-1]]
    vol = np.random.lognormal(3, .5, n)*(1+50*np.abs(r))
    return pd.DataFrame({"timestamp": idx, "open": open_, "high": high, "low": low, "close": close, "volume": vol})

if os.path.exists(DATA_PATH):
    df = pd.read_csv(DATA_PATH).rename(columns=COLUMN_MAP)
    df.columns = [c.lower() for c in df.columns]
    if ASSET_COL: df = df[df[ASSET_COL.lower()] == ASSET]
    ts = df["timestamp"]
    df["timestamp"] = pd.to_datetime(ts, unit="s") if np.issubdtype(ts.dtype, np.number) else pd.to_datetime(ts)
    print("Loaded real data:", DATA_PATH)
else:
    df = synthetic(); print("DATA NOT FOUND -> using synthetic data")
df = df.sort_values("timestamp").drop_duplicates("timestamp").set_index("timestamp")[["open","high","low","close","volume"]].astype(float)
df = df.dropna()
print(df.shape, df.index.min(), "->", df.index.max())
df.tail()

## 2. Feature engineering
Momentum (returns, RSI, MACD), trend (EMA gaps), volatility (rolling std, range), and volume (spike ratio). All use only past data.

In [ ]:
def rsi(s, n=14):
    d = s.diff(); up = d.clip(lower=0).ewm(alpha=1/n, adjust=False).mean(); dn = (-d.clip(upper=0)).ewm(alpha=1/n, adjust=False).mean()
    return 100 - 100/(1 + up/(dn + 1e-12))

def make_features(d):
    f = pd.DataFrame(index=d.index); c = d["close"]
    for k in (1, 4, 16, 96):                       # 15m, 1h, 4h, 24h
        f[f"ret_{k}"] = c.pct_change(k)
    f["rsi_14"] = rsi(c)
    ema12, ema26 = c.ewm(span=12, adjust=False).mean(), c.ewm(span=26, adjust=False).mean()
    f["macd"] = (ema12 - ema26)/c
    f["macd_signal"] = f["macd"].ewm(span=9, adjust=False).mean()
    f["macd_hist"] = f["macd"] - f["macd_signal"]
    for s in (20, 50, 200):
        f[f"ema_gap_{s}"] = c/c.ewm(span=s, adjust=False).mean() - 1
    r1 = c.pct_change()
    for w in (16, 96):
        f[f"vol_{w}"] = r1.rolling(w).std()
    f["range_hl"] = (d["high"] - d["low"])/c
    f["volume_spike"] = d["volume"]/d["volume"].rolling(96).mean()
    f["hour_sin"] = np.sin(2*np.pi*d.index.hour/24); f["hour_cos"] = np.cos(2*np.pi*d.index.hour/24)
    return f

X = make_features(df)
fwd_ret = df["close"].shift(-HORIZON_BARS)/df["close"] - 1      # 24h forward return
data = X.assign(fwd_ret=fwd_ret, y_dir=(fwd_ret > 0).astype(int)).replace([np.inf, -np.inf], np.nan).dropna()
FEATURES = list(X.columns)
print(data.shape, "| positive-direction rate:", data["y_dir"].mean().round(3))

## 3. Chronological split with embargo
No shuffling. A gap of one horizon (96 bars) between sets prevents the overlapping 24h target windows from leaking.

In [ ]:
n = len(data); gap = HORIZON_BARS
i_tr, i_va = int(n*0.70), int(n*0.85)
tr, va, te = data.iloc[:i_tr-gap], data.iloc[i_tr:i_va-gap], data.iloc[i_va:]
print(len(tr), len(va), len(te))
sc = StandardScaler().fit(tr[FEATURES])
Xtr, Xva, Xte = (sc.transform(s[FEATURES]) for s in (tr, va, te))
def report(name, p_up, ret_pred, s):
    acc = accuracy_score(s["y_dir"], (p_up > .5).astype(int)); auc = roc_auc_score(s["y_dir"], p_up)
    mae = mean_absolute_error(s["fwd_ret"], ret_pred)
    return {"model": name, "accuracy": acc, "auc": auc, "MAE": mae}
results = []

## 4. Baselines
Random walk (predict 0 return, majority direction) and a naïve momentum rule, plus logistic/ridge linear baselines.

In [ ]:
maj = int(tr["y_dir"].mean() > .5)
results.append(report("random-walk / majority", np.full(len(te), .5+(maj-.5)*.01), np.zeros(len(te)), te))
results.append(report("naive momentum (sign of past 24h)", (te["ret_96"] > 0).astype(float).values*.98+.01, te["ret_96"].values*0, te))
lr = LogisticRegression(max_iter=1000).fit(Xtr, tr["y_dir"]); rg = Ridge(alpha=10).fit(Xtr, tr["fwd_ret"])
results.append(report("logistic + ridge", lr.predict_proba(Xte)[:,1], rg.predict(Xte), te))
pd.DataFrame(results)

## 5. XGBoost (classifier + regressor)

In [ ]:
xgb_c = XGBClassifier(n_estimators=400, max_depth=3, learning_rate=0.03, subsample=.8, colsample_bytree=.8, eval_metric="logloss", early_stopping_rounds=30, random_state=RNG)
xgb_r = XGBRegressor(n_estimators=400, max_depth=3, learning_rate=0.03, subsample=.8, colsample_bytree=.8, early_stopping_rounds=30, random_state=RNG)
xgb_c.fit(Xtr, tr["y_dir"], eval_set=[(Xva, va["y_dir"])], verbose=False)
xgb_r.fit(Xtr, tr["fwd_ret"], eval_set=[(Xva, va["fwd_ret"])], verbose=False)
p_xgb, r_xgb = xgb_c.predict_proba(Xte)[:,1], xgb_r.predict(Xte)
results.append(report("XGBoost", p_xgb, r_xgb, te))
pd.DataFrame(results)

## 6. GRU on 15-minute sequence windows
Requires PyTorch (`pip install torch`). Skipped gracefully if unavailable.

In [ ]:
p_gru = r_gru = None
try:
    import torch, torch.nn as nn
    torch.manual_seed(RNG)
    def windows(Xs, idx_df):
        # window ending at each row, using the scaled feature history of the full series
        full = pd.DataFrame(sc.transform(data[FEATURES]), index=data.index)
        pos = data.index.get_indexer(idx_df.index)
        ok = pos >= SEQ_LEN
        arr = np.stack([full.values[p-SEQ_LEN+1:p+1] for p in pos[ok]])
        return torch.tensor(arr, dtype=torch.float32), ok
    class Net(nn.Module):
        def __init__(s, d):
            super().__init__(); s.gru = nn.GRU(d, 32, batch_first=True); s.cls = nn.Linear(32, 1); s.reg = nn.Linear(32, 1)
        def forward(s, x):
            h = s.gru(x)[0][:, -1]; return s.cls(h).squeeze(-1), s.reg(h).squeeze(-1)
    Wtr, mtr = windows(Xtr, tr); Wva, mva = windows(Xva, va); Wte, mte = windows(Xte, te)
    ytr_c = torch.tensor(tr["y_dir"].values[mtr], dtype=torch.float32); ytr_r = torch.tensor(tr["fwd_ret"].values[mtr]*10, dtype=torch.float32)
    net = Net(len(FEATURES)); opt = torch.optim.Adam(net.parameters(), 1e-3)
    best, bad = 1e9, 0
    for ep in range(30):
        net.train(); perm = torch.randperm(len(Wtr))
        for b in perm.split(256):
            lc, lr_ = net(Wtr[b]); loss = nn.functional.binary_cross_entropy_with_logits(lc, ytr_c[b]) + nn.functional.mse_loss(lr_, ytr_r[b])
            opt.zero_grad(); loss.backward(); opt.step()
        net.eval()
        with torch.no_grad():
            lc, lr_ = net(Wva); vloss = (nn.functional.binary_cross_entropy_with_logits(lc, torch.tensor(va["y_dir"].values[mva], dtype=torch.float32)) ).item()
        if vloss < best - 1e-4: best, bad, state = vloss, 0, {k: v.clone() for k, v in net.state_dict().items()}
        else:
            bad += 1
            if bad >= 5: break
    net.load_state_dict(state); net.eval()
    with torch.no_grad():
        lc, lr_ = net(Wte)
    te_g = te[mte]; p_gru, r_gru = torch.sigmoid(lc).numpy(), lr_.numpy()/10
    results.append(report("GRU (test rows with full window)", p_gru, r_gru, te_g))
except ImportError:
    print("PyTorch not installed - skipping GRU")
pd.DataFrame(results)

## 7. Ensemble
Average of logistic, XGBoost (and GRU when available) probabilities / return predictions.

In [ ]:
ps, rs = [lr.predict_proba(Xte)[:,1], p_xgb], [rg.predict(Xte), r_xgb]
te_e = te
if p_gru is not None:
    keep = np.isin(te.index, te_g.index)
    ps = [p[keep] for p in ps] + [p_gru]; rs = [r[keep] for r in rs] + [r_gru]; te_e = te_g
results.append(report("Ensemble (avg)", np.mean(ps, 0), np.mean(rs, 0), te_e))
res = pd.DataFrame(results).set_index("model").round(4); res

In [ ]:
res[["accuracy"]].plot.barh(legend=False, figsize=(7, 3.5)); plt.axvline(.55, color="gray", ls="--", label="55% target")
plt.title("24h directional accuracy on held-out test set"); plt.xlabel("accuracy"); plt.tight_layout(); plt.show()

## 8. Interpretability: permutation importance & SHAP

In [ ]:
pi = permutation_importance(xgb_c, Xte, te["y_dir"], n_repeats=10, random_state=RNG, scoring="roc_auc")
imp = pd.Series(pi.importances_mean, index=FEATURES).sort_values()
imp.plot.barh(figsize=(7, 5)); plt.title("Permutation importance (drop in test AUC)"); plt.tight_layout(); plt.show()

In [ ]:
try:
    import shap
    sv = shap.TreeExplainer(xgb_c).shap_values(Xte[:2000])
    shap.summary_plot(sv, pd.DataFrame(Xte[:2000], columns=FEATURES))
except ImportError:
    print("pip install shap to enable SHAP plots")

## 9. Conclusions TBD 